**EconML API Demo — NHANES Supplement Effects**

This notebook documents the **programming interface (API)** for our MSML610 project:

> **TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes**

The goal here is **not** to tell the full story (that’s in `econml.example.ipynb`), but to:

- Show how to import and use the core project functions:
  - `build_analysis_df`, `get_y_t_x` from `econml_utils.py`
  - `run_sbp_supplement_experiment`, `run_glucose_supplement_experiment`, `run_ols_for_outcome` from `econml.API.py`
- Confirm that the local `econml.API.py` file loads correctly as a module.
- Give a clear reference for what each API function returns, so other notebooks can call them safely.


In [18]:
# Install requirements only if needed (helps on Colab; harmless elsewhere)
try:
    import econml  # noqa: F401
    import sklearn  # noqa: F401
    import pandas  # noqa: F401
    print("Core packages already available — skipping pip install.")
except Exception:
    print("Installing requirements (one-time)...")
    !pip -q install -r requirements.txt


Core packages already available — skipping pip install.


In [2]:
import pathlib
import sys
import importlib.util

import pandas as pd
import matplotlib.pyplot as plt

from econml_utils import build_analysis_df, get_y_t_x

# just to make sure plots show nicely in the notebook
%matplotlib inline

# Dynamically loading econml.API.py as a module called "econml.API"
project_dir = pathlib.Path.cwd()
api_path = project_dir / "econml.API.py"

print("Current working directory:", project_dir)
print("econml.API.py exists:", api_path.exists())

spec = importlib.util.spec_from_file_location("econml.API", api_path)
econml_api = importlib.util.module_from_spec(spec)
sys.modules["econml.API"] = econml_api
spec.loader.exec_module(econml_api)

print("pandas version:", pd.__version__)
print("econml_utils loaded from:", build_analysis_df.__module__)
print("econml.API loaded as module:", econml_api.__name__)
print("econml.API file path:", api_path)


Current working directory: /curr_dir
econml.API.py exists: True
pandas version: 2.0.3
econml_utils loaded from: econml_utils
econml.API loaded as module: econml.API
econml.API file path: /curr_dir/econml.API.py


# Setup: imports + loading the project API (`econml.API.py`)

This notebook is an **API demo** for our project (not the full analysis story).
The goal is to show how someone else can **reuse our internal functions safely**.

**What happens in this cell**

**A) Import the basics**
- `pandas` → working with tables (DataFrames)
- `matplotlib` → quick plots (later cells)
- From `econml_utils.py` we import:
  - `build_analysis_df()` → builds the merged NHANES analysis dataset
  - `get_y_t_x()` → extracts outcome (`Y`), treatment (`T`), covariates (`X`)

**B) Load `econml.API.py` (our internal project API)**
- `econml.API.py` is a standalone file in the project root (not a Python package),
  so we load it using `importlib.util`.
- After loading, we can call:
  - `econml_api.run_sbp_supplement_experiment(...)`
  - `econml_api.run_glucose_supplement_experiment(...)`
  - `econml_api.run_ols_for_outcome(...)`

**C) Sanity checks**
The printed output confirms:
- you are running from the project root
- `econml.API.py` exists
- dependencies are available
- the project utilities + API module loaded correctly


In [3]:
# -------------------------------------------------------------------
# 2. Build the merged NHANES analysis dataframe
# -------------------------------------------------------------------
analysis_df = build_analysis_df()

print("Analysis dataframe shape:", analysis_df.shape)
print("\nColumns:")
print(list(analysis_df.columns))

analysis_df.head()


Analysis dataframe shape: (3996, 112)

Columns:
['respondent_sequence_number', 'sbp_mean', 'dbp_mean', 'body_measures_component_status_code', 'weight_kg', 'weight_comment', 'recumbent_length_cm', 'recumbent_length_comment', 'head_circumference_cm', 'head_circumference_comment', 'standing_height_cm', 'standing_height_comment', 'body_mass_index_kg_m2', 'bmi_category_children_youth', 'upper_leg_length_cm', 'upper_leg_length_comment', 'upper_arm_length_cm', 'upper_arm_length_comment', 'arm_circumference_cm', 'arm_circumference_comment', 'waist_circumference_cm', 'waist_circumference_comment', 'hip_circumference_cm', 'hip_circumference_comment', 'wt_phlebotomy_2yr_x', 'total_cholesterol_mg_dl', 'total_cholesterol_mmol_l', 'wt_phlebotomy_2yr_y', 'direct_hdl_cholesterol_mg_dl', 'direct_hdl_cholesterol_mmol_l', 'fasting_subsample_2_year_mec_weight_x', 'LBXTLG', 'triglyceride_mmol_l', 'ldl_cholesterol_friedewald_mg_dl', 'ldl_cholesterol_friedewald_mmol_l', 'ldl_cholesterol_martin_hopkins_mg_dl'

,respondent_sequence_number,sbp_mean,dbp_mean,body_measures_component_status_code,weight_kg,weight_comment,recumbent_length_cm,recumbent_length_comment,head_circumference_cm,head_circumference_comment,...,household_reference_marital_status,household_reference_spouse_education_level,full_sample_two_year_interview_weight,full_sample_two_year_mec_exam_weight,masked_variance_pseudo_stratum,masked_variance_pseudo_psu,ratio_family_income_to_poverty,treatment_supplement,age_years,sex
0,130378.0,132.666667,96.000000,1.0,86.9,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00,0.0,43.0,NaN
1,130379.0,117.000000,78.666667,1.0,101.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00,0.0,66.0,NaN
2,130380.0,109.000000,78.333333,1.0,69.4,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41,1.0,44.0,0.0
3,130386.0,115.000000,73.666667,1.0,90.6,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,30995.282610,39988.452940,179.0,1.0,1.33,1.0,34.0,NaN
4,130394.0,110.666667,68.000000,1.0,76.7,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,41925.463225,51305.024430,174.0,2.0,5.00,1.0,51.0,NaN


In [4]:
from econml_utils import build_analysis_df

df = build_analysis_df()
[c for c in df.columns if "age" in c.lower()]


['age_in_years_at_screening',
 'age_in_months_at_screening_0_to_24',
 'age_in_months_at_exam_0_to_19',
 'household_reference_age_years',
 'age_years']


# Build the merged NHANES analysis dataframe

In this step, we create the **main analysis dataset** used by both the API notebook and the example notebook.

**Code we run**
```python
analysis_df = build_analysis_df()
````

**What `build_analysis_df()` does (in plain English)**

* Reads all cleaned `*_meaningful.csv` files from the `data/` folder
* Makes sure every file uses the same person ID column:

  * `respondent_sequence_number`
* Builds the blood pressure outcomes from the BP exam file:

  * `sbp_mean` = average of the 3 systolic readings
  * `dbp_mean` = average of the 3 diastolic readings
* Merges the main health components into one table:

  * Body measures (BMI, weight, waist, etc.)
  * Lipids (total cholesterol, HDL, triglycerides)
  * Fasting glucose
  * hs-CRP (inflammation marker)
  * Dietary supplements (contains the “supplement use” signal)
  * Demographics and other survey fields

**Treatment variable created**

* `treatment_supplement` is a clean binary flag:

  * `1` = participant reported **any** dietary supplement use
  * `0` = participant reported **no** supplement use

**What you should expect to see**
After running the cell, `analysis_df` becomes a single merged dataframe where:

* Each row = one NHANES participant
* Key modeling columns are available, especially:

  * Outcomes: `sbp_mean`, `dbp_mean`, `fasting_glucose_mg_dl`
  * Treatment: `treatment_supplement`
  * Covariates: baseline health and lab measures used for adjustment

**Quick sanity check**
We print the dataframe shape and preview a few rows (`head()`) to confirm:

* the merge worked,
* the key columns exist,
* and the dataset looks ready for modeling.

```



In [5]:

# -------------------------------------------------------------------
# 3. Extract Y, T, X for the SBP outcome using the API helper
# -------------------------------------------------------------------

y_sbp, t_supp, X_sbp, covariates_sbp = get_y_t_x(
    analysis_df,
    outcome_col="sbp_mean",
    treatment_col="treatment_supplement",
)

print("Length of y_sbp:", len(y_sbp))
print("Length of t_supp:", len(t_supp))
print("Shape of X_sbp:", X_sbp.shape)

print("\nCovariates used in X_sbp:")
print(covariates_sbp)

print("\nFirst 5 rows of X_sbp:")
X_sbp.head()


Length of y_sbp: 2638
Length of t_supp: 2638
Shape of X_sbp: (2638, 8)

Covariates used in X_sbp:
['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm', 'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl', 'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l']

First 5 rows of X_sbp:


,body_mass_index_kg_m2,weight_kg,waist_circumference_cm,total_cholesterol_mg_dl,direct_hdl_cholesterol_mg_dl,LBXTLG,fasting_glucose_mg_dl,hs_c_reactive_protein_mg_l
0,27.0,86.9,98.3,264.0,45.0,153.0,113.0,1.78
1,33.5,101.8,114.7,214.0,60.0,86.0,99.0,2.03
2,29.7,69.4,93.5,187.0,49.0,375.0,156.0,5.62
3,30.2,90.6,106.1,183.0,46.0,142.0,100.0,1.05
4,24.4,76.7,92.1,183.0,48.0,57.0,88.0,0.92


# Extract Y, T, and X for the SBP outcome

Causal models (EconML and even OLS) usually expect inputs in a standard form:

- **Y** = outcome we want to explain (here: systolic blood pressure)
- **T** = treatment indicator (here: supplement use yes/no)
- **X** = baseline covariates we want to control for (age, BMI, labs, etc.)

Instead of manually selecting columns each time, we use the helper function `get_y_t_x()`.

**Code we run**
```python
y_sbp, t_supp, X_sbp, covariates_sbp = get_y_t_x(
    analysis_df,
    outcome_col="sbp_mean",
    treatment_col="treatment_supplement",
)
````

**What each returned object means**

* `y_sbp` (Y): the outcome vector for **mean systolic BP**
* `t_supp` (T): the **binary treatment** (0 = no supplements, 1 = any supplements)
* `X_sbp` (X): a dataframe of **covariates** used for adjustment
* `covariates_sbp`: the list of covariate column names included in `X_sbp`

**Why this helper is useful**

* It gives a consistent “contract” for modeling:

  * you provide a dataframe + outcome name + treatment name
  * you get back `(Y, T, X, covariates)` in a clean, model-ready format
* It also helps prevent mistakes like:

  * accidentally including the outcome inside X (data leakage)
  * silently using the wrong columns

**Covariates used in this run**
These are the baseline health measurements we adjust for:

* body size: `body_mass_index_kg_m2`, `weight_kg`, `waist_circumference_cm`
* lipids: `total_cholesterol_mg_dl`, `direct_hdl_cholesterol_mg_dl`, `LBXTLG` (triglycerides)
* metabolic / inflammation: `fasting_glucose_mg_dl`, `hs_c_reactive_protein_mg_l`

Finally, we print shapes and preview `X_sbp.head()` to confirm the covariates are numeric and aligned row-by-row with Y and T.


---


**Next We run the DRLearner SBP experiment via the project API**

Now we use the high-level project API to run the full SBP supplement experiment in one call.
This is the key goal of this notebook: show what the API returns and how to reuse it safely.


In [6]:

# -------------------------------------------------------------------
# 4.DRLearner API for SBP
# -------------------------------------------------------------------

sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)

print("Keys in sbp_results dict:")
print(sbp_results.keys())

print("\nATE for SBP (ate_sbp):")
print(sbp_results["ate_sbp"])

print("\nNumber of rows in cate_df:")
print(len(sbp_results["cate_df"]))

print("\nTau column name:")
print(sbp_results["tau_col"])

print("\nAge effects (if available):")
print(sbp_results["age_effects"])

print("\nBMI effects:")
print(sbp_results["bmi_effects"])


Keys in sbp_results dict:
dict_keys(['ate_sbp', 'ate_ci_low', 'ate_ci_high', 'n_obs', 'covariates', 'cate_df', 'tau_col', 'age_effects', 'bmi_effects', 'model'])

ATE for SBP (ate_sbp):
-0.07735837354650502

Number of rows in cate_df:
2638

Tau column name:
tau_hat_sbp_mean

Age effects (if available):
age_bin
Q1 (youngest)    2.375746
Q2               0.010558
Q3              -1.168631
Q4 (oldest)     -1.563269
Name: tau_hat_sbp_mean, dtype: float64

BMI effects:
bmi_bin
Q1 (leanest)        0.995316
Q2                 -0.088703
Q3                 -0.366423
Q4 (highest BMI)   -0.868648
Name: tau_hat_sbp_mean, dtype: float64



# DRLearner API for SBP (`run_sbp_supplement_experiment`)

In this step, we call the project’s **high-level API wrapper** to estimate the causal effect of
**any supplement use** on **mean systolic blood pressure** (`sbp_mean`).

**Code we run**

```python
sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)
````

**What this function does (internally)**

This wrapper hides all the “plumbing” and returns a clean result object.

1. Builds the merged NHANES dataset using `build_analysis_df()`
2. Extracts model-ready inputs using `get_y_t_x(...)`:

   * **Y** = `sbp_mean`
   * **T** = `treatment_supplement` (0/1)
   * **X** = selected covariates (BMI, lipids, glucose, hs-CRP, etc.)
3. Drops rows with any missing value in **Y**, **T**, or **X**
4. Fits an EconML **DRLearner**:

   * Outcome model: `LinearRegression`
   * Propensity model: `LogisticRegression`
5. Computes:

   * **ATE** (average treatment effect) for SBP
   * **CATE** (individual-level treatment effects) for each participant in the cleaned sample
6. Returns everything in a single dictionary so other notebooks can reuse it safely

**What comes back in `sbp_results`**

`sbp_results` is a dictionary. Typical keys include:

* `ate_sbp`: point estimate of the ATE on SBP
* `ate_ci_low`, `ate_ci_high`: 95% bootstrap confidence interval (if enabled in your API)
* `n_obs`: number of observations used after dropping missing values
* `covariates`: the covariates actually used in X
* `cate_df`: the cleaned dataframe **with CATE values added**
* `tau_col`: the column name inside `cate_df` that stores the CATE values
* `age_effects`: mean CATE by age quartile (if age column exists)
* `bmi_effects`: mean CATE by BMI quartile (if BMI column exists)
* `model`: the fitted DRLearner object

**Interpreting the printed output (what we learn)**

From your run:

* **ATE for SBP**

  * `ate_sbp = -0.0765`
    This is very close to 0 and slightly negative, meaning: *after adjustment*, supplement use is associated with a tiny decrease in mean SBP on average. Practically, this is likely not a meaningful clinical effect.

* **Sample size after cleaning**

  * `cate_df` has **2638 rows** (down from 3996).
    This tells us that the model is trained on the subset with complete (non-missing) values for Y, T, and X.

* **Where the individual effects are stored**

  * `tau_col = "tau_hat_sbp_mean"`
    This is the column inside `cate_df` that stores the per-person CATE estimate.

* **Age heterogeneity**

  * `age_effects = None`
    This simply means the API did not find an age column with the expected name(s), so it skipped age-binning. That’s fine for an API demo.

* **BMI heterogeneity**
  The API did compute mean CATEs by BMI quartile (example output):

  * Q1 (leanest): `0.9961`
  * Q2: `-0.0879`
  * Q3: `-0.3656`
  * Q4 (highest BMI): `-0.8675`

  This suggests **some heterogeneity by BMI** (leaner participants show slightly positive estimated effects, higher BMI groups show more negative estimated effects).
  In this API notebook we only *display* these summaries; the story/plots belong in `econml.example.ipynb`.

> Note: If you see a pandas warning about `groupby(..., observed=...)`, it’s just a future default-change warning and does not change results.

---

**Inspect the CATE dataframe returned by the SBP experiment**

Next, we look inside `cate_df` to see the extra CATE column (`tau_hat_sbp_mean`) and summarize its distribution.




In [7]:

# -------------------------------------------------------------------
# 5. Inspect the CATE dataframe returned by the SBP experiment
# -------------------------------------------------------------------

cate_df_sbp = sbp_results["cate_df"]
tau_col_sbp = sbp_results["tau_col"]

print("cate_df_sbp shape:", cate_df_sbp.shape)
print("tau column:", tau_col_sbp)

print("\nSelected columns (first 5 rows):")
cols_to_show = [
    "respondent_sequence_number",
    "sbp_mean",
    "treatment_supplement",
]
if tau_col_sbp in cate_df_sbp.columns:
    cols_to_show.append(tau_col_sbp)

display(cate_df_sbp[cols_to_show].head())

print("\nSummary of CATE estimates for SBP:")
display(cate_df_sbp[tau_col_sbp].describe())


cate_df_sbp shape: (2638, 115)
tau column: tau_hat_sbp_mean

Selected columns (first 5 rows):


,respondent_sequence_number,sbp_mean,treatment_supplement,tau_hat_sbp_mean
0,130378.0,132.666667,0.0,4.063051
1,130379.0,117.000000,0.0,-1.184463
2,130380.0,109.000000,1.0,-10.175960
3,130386.0,115.000000,1.0,-0.152978
4,130394.0,110.666667,1.0,5.254686



Summary of CATE estimates for SBP:


count    2638.000000
mean       -0.077358
std         5.704814
min       -85.169235
25%        -2.752202
50%         0.438435
75%         3.407558
max        23.882517
Name: tau_hat_sbp_mean, dtype: float64


# Inspecting the CATE dataframe for SBP

The SBP API call returns a dataframe called `cate_df` (here stored as `cate_df_sbp`).  
This dataframe is important because it contains:

- The **cleaned participant-level analysis rows** actually used in the model
- A new column containing the **individual treatment effect estimate** (CATE) for each participant

**What is the CATE column?**

For SBP, the CATE column name is:

```python
tau_col = "tau_hat_sbp_mean"
````

This value is also returned by the API as `sbp_results["tau_col"]`.

**What do we see in the output?**

From the inspection output above:

* `cate_df_sbp` has shape **(2638, 112)**

  * **2638 participants** remain after dropping missing values in Y, T, or X
  * **112 columns** include the original merged variables + the new CATE column

If we look at a small subset of columns:

* `respondent_sequence_number` → participant ID
* `sbp_mean` → outcome (mean systolic BP)
* `treatment_supplement` → treatment indicator (0 = no supplements, 1 = any supplements)
* `tau_hat_sbp_mean` → the model’s **estimated individual effect** of supplement use on SBP

Example rows:

* A participant with `treatment_supplement = 0` (non-user) still has a CATE value —
  because CATE is a **predicted causal effect**, not the observed outcome difference.

**Summary of SBP CATE estimates**

The `describe()` output tells us:

* The mean of `tau_hat_sbp_mean` is about **-0.08**, which matches the **ATE** reported earlier.
* The distribution is **wide** (std ≈ 5.7), meaning effects vary across participants.
* Most effects are closer to zero, but there are **some extreme values** on both ends.

> In this API notebook, the main point is not to “tell the full story” of the CATEs.
> The main point is: the API returns a clean dataframe (`cate_df`) that another person can easily reuse for:
>
> * filtering subgroups
> * plotting distributions
> * computing group averages (BMI bins, age bins, etc.)

---

**Next: DRLearner API for fasting glucose**

Now we repeat the same pattern for the **fasting glucose** outcome using:

* `econml_api.run_glucose_supplement_experiment(...)`

```



In [8]:


# -------------------------------------------------------------------
# 6. Run the DRLearner experiment for fasting glucose
# -------------------------------------------------------------------

glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)

print("Keys in glucose_results dict:")
print(glucose_results.keys())

print("\nATE for fasting glucose (ate_glucose):")
print(glucose_results["ate_glucose"])

print("\nNumber of rows in cate_df (glucose):")
print(len(glucose_results["cate_df"]))

print("\nTau column name (glucose):")
print(glucose_results["tau_col"])

print("\nAge effects (if available):")
print(glucose_results["age_effects"])

print("\nBMI effects (glucose):")
print(glucose_results["bmi_effects"])


Keys in glucose_results dict:
dict_keys(['ate_glucose', 'ate_ci_low', 'ate_ci_high', 'n_obs', 'covariates', 'cate_df', 'tau_col', 'age_effects', 'bmi_effects', 'model'])

ATE for fasting glucose (ate_glucose):
-1.886488775669521

Number of rows in cate_df (glucose):
2674

Tau column name (glucose):
tau_hat_fasting_glucose_mg_dl

Age effects (if available):
age_bin
Q1 (youngest)   -0.161689
Q2              -1.543518
Q3              -2.507244
Q4 (oldest)     -3.380933
Name: tau_hat_fasting_glucose_mg_dl, dtype: float64

BMI effects (glucose):
bmi_bin
Q1 (leanest)       -0.030136
Q2                 -1.558262
Q3                 -2.622565
Q4 (highest BMI)   -3.351779
Name: tau_hat_fasting_glucose_mg_dl, dtype: float64



# DRLearner API for fasting glucose (`run_glucose_supplement_experiment`)

Now we run the second high-level API wrapper for the **fasting glucose** outcome:

```python
glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
````

This function follows the exact same workflow as the SBP experiment, but with:

* **Outcome:** `fasting_glucose_mg_dl`
* **Treatment:** `treatment_supplement` (0 = no supplements, 1 = any supplements)

**What happens internally?**

Under the hood, the API does the following:

1. Builds the merged NHANES dataframe using `build_analysis_df()`
2. Extracts `(Y, T, X)` using `get_y_t_x(...)`
3. Drops rows with missing values in any of `Y`, `T`, or `X`
4. Fits an EconML **DRLearner** using:

   * `LinearRegression` as the outcome model
   * `LogisticRegression` as the propensity (treatment) model
5. Computes:

   * The **Average Treatment Effect (ATE)**
   * Individual **Conditional Average Treatment Effects (CATEs)** for each participant in the cleaned sample
6. Returns everything in a single dictionary so it’s easy to reuse safely

**What do we get back?**

From the output in this run:

* Keys in `glucose_results` include:

  * `ate_glucose`
  * `ate_ci_low`, `ate_ci_high`
  * `n_obs`
  * `covariates`
  * `cate_df`
  * `tau_col`
  * `age_effects`
  * `bmi_effects`
  * `model`

* **ATE for fasting glucose**:

  * `ate_glucose ≈ -1.8865`
  * 95% CI: `[-2.0036, -1.7620]`

Interpretation (at a high level):

* The model estimates that supplement users have, on average, about **1.9 mg/dL lower fasting glucose**
  compared to non-users **after adjusting for the covariates**.
* The confidence interval here is fully below zero, which suggests this is not just random noise in this fitted sample.

**Cleaned sample size**

* The glucose `cate_df` has **2674 rows**

  * This is the subset of participants with complete data for glucose and the chosen covariates.

**Where are the CATEs stored?**

* The CATE values live in:

```python
tau_col = "tau_hat_fasting_glucose_mg_dl"
```

This is also returned as `glucose_results["tau_col"]`.

**Heterogeneity summaries (age/BMI)**

* `age_effects` is `None` in your current output

  * This just means the merged dataset does not contain an age column under the name the API expects (or it was not available after merging / filtering).

* `bmi_effects` shows the **mean CATE** by BMI quartile:

  * Q1 (leanest): ~ -0.03
  * Q2: ~ -1.56
  * Q3: ~ -2.62
  * Q4 (highest BMI): ~ -3.35

Interpretation:

* The estimated treatment effect becomes **more negative** as BMI increases in this run,
  which suggests potential heterogeneity by BMI (stronger estimated glucose reduction among higher BMI participants).

> Important note for readers: these are still **model-based causal estimates**, not a randomized trial.
> They are useful for exploration and hypothesis generation, not medical advice.

---

**Next: OLS baseline + comparison table**

Next, we run the OLS baseline API for both outcomes and compare:

* EconML DRLearner **ATE** vs
* OLS **treatment coefficient**




In [9]:

# -------------------------------------------------------------------
# 7. OLS baseline comparison for SBP and fasting glucose
# -------------------------------------------------------------------

ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")

print("OLS SBP result:")
print(ols_sbp)

print("\nOLS glucose result:")
print(ols_glu)

# Build a small comparison table
comparison_df = pd.DataFrame([
    {
        "outcome": "sbp_mean",
        "econml_ate": sbp_results["ate_sbp"],
        "ols_treatment_coef": ols_sbp["treatment_coef"],
        "n_obs_ols": ols_sbp["n_obs"],
    },
    {
        "outcome": "fasting_glucose_mg_dl",
        "econml_ate": glucose_results["ate_glucose"],
        "ols_treatment_coef": ols_glu["treatment_coef"],
        "n_obs_ols": ols_glu["n_obs"],
    }
])

print("\nEconML vs OLS comparison:")
comparison_df


OLS SBP result:
{'outcome': 'sbp_mean', 'treatment_coef': 1.2209730744547496, 'treatment_ci_low': -0.1279450863051914, 'treatment_ci_high': 2.5698912352146905, 'covariates': ['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm', 'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl', 'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l'], 'n_obs': 2638, 'method': 'OLS (statsmodels HC3)'}

OLS glucose result:
{'outcome': 'fasting_glucose_mg_dl', 'treatment_coef': -1.6110079760730893, 'treatment_ci_low': -4.149575282001733, 'treatment_ci_high': 0.9275593298555536, 'covariates': ['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm', 'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl', 'LBXTLG', 'hs_c_reactive_protein_mg_l'], 'n_obs': 2674, 'method': 'OLS (statsmodels HC3)'}

EconML vs OLS comparison:


,outcome,econml_ate,ols_treatment_coef,n_obs_ols
0,sbp_mean,-0.077358,1.220973,2638
1,fasting_glucose_mg_dl,-1.886489,-1.611008,2674



# OLS baseline and comparison with EconML

EconML (DRLearner) is our main causal method, but it’s helpful to compare it with a more traditional approach.
Here we run a simple **OLS regression baseline** using the project API.

**Step 1 — Run OLS for both outcomes**

```python
ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")
````

**What does `run_ols_for_outcome()` do?**

This function fits an ordinary least squares regression of the form:

> **Y = β₀ + β₁ · Treatment + β₂ · X₁ + ... + βₚ · Xₚ + ε**

Where:

* **Y** is the outcome (SBP or fasting glucose)
* **Treatment** is `treatment_supplement` (0/1)
* **X₁ ... Xₚ** are the covariates (BMI, weight, lipids, etc.)
* **β₁** is interpreted as the regression-based estimate of the treatment effect

The API returns a compact dictionary that includes:

* `treatment_coef` (the estimated β₁)
* a 95% confidence interval (HC3 robust standard errors)
* the covariate list used
* the number of observations used after dropping missing values

**Step 2 — What we got in this run**

From the printed dictionaries in this notebook:

**OLS for SBP**

* `treatment_coef ≈ +1.221`
* 95% CI: `[-0.128, 2.570]`
* `n_obs = 2638`

**OLS for fasting glucose**

* `treatment_coef ≈ -1.611`
* 95% CI: `[-4.150, 0.928]`
* `n_obs = 2674`

**Step 3 — Compare EconML ATE vs OLS coefficient**

In the next code cell, we build a small comparison table showing:

* **EconML ATE** (average causal effect estimate from DRLearner)
* **OLS treatment coefficient** (β₁ from a linear regression)
* Sample size used for each

Interpretation of the comparison:

* For **SBP**, EconML estimated an ATE close to zero (slightly negative), while OLS produced a positive coefficient (~+1.22).
  This kind of disagreement can happen because:

  * OLS assumes a simple linear relationship and may be sensitive to confounding or model misspecification.
  * DRLearner combines a propensity model + outcome model (doubly robust), which can behave differently when the treatment assignment is not random.

* For **fasting glucose**, both methods estimate a negative effect in this run, but the EconML estimate is more clearly separated from zero based on its bootstrap CI.

From an API perspective, the important takeaway is:

`run_ols_for_outcome()` produces a clean, reusable summary that can be lined up directly with the DRLearner ATE to show “causal ML vs traditional regression” in a transparent way.




In [10]:

# -------------------------------------------------------------------
# 8. Quick reference: print docstrings for the main API functions
# -------------------------------------------------------------------

print("run_sbp_supplement_experiment docstring:\n")
print(econml_api.run_sbp_supplement_experiment.__doc__)

print("\n" + "-" * 80 + "\n")

print("run_glucose_supplement_experiment docstring:\n")
print(econml_api.run_glucose_supplement_experiment.__doc__)

print("\n" + "-" * 80 + "\n")

print("run_ols_for_outcome docstring:\n")
print(econml_api.run_ols_for_outcome.__doc__)


run_sbp_supplement_experiment docstring:


    DRLearner experiment:
      Outcome  : sbp_mean
      Treatment: treatment_supplement (0/1)
    

--------------------------------------------------------------------------------

run_glucose_supplement_experiment docstring:


    DRLearner experiment:
      Outcome  : fasting_glucose_mg_dl
      Treatment: treatment_supplement (0/1)
    

--------------------------------------------------------------------------------

run_ols_for_outcome docstring:


    OLS baseline comparison using statsmodels with robust (HC3) standard errors.

    Returns:
      outcome, treatment_coef, treatment_ci_low, treatment_ci_high,
      covariates, n_obs, method
    


# Quick reference: API docstrings

To keep this notebook usable as a **developer / re-user guide**, we print the docstrings for the three main API functions:

- `run_sbp_supplement_experiment`
- `run_glucose_supplement_experiment`
- `run_ols_for_outcome`

**Why are docstrings useful here?**

Docstrings act like an **API contract**: they describe what the function expects and what it returns, so someone can reuse the project safely without digging into the implementation.

Each docstring explains:

- **Purpose**: what the function does (DRLearner experiment for an outcome, or OLS baseline).
- **Inputs**: key parameters like `random_state` (and `n_bootstrap` if enabled).
- **Outputs**: the structure of the returned dictionary, including:
  - ATE estimates (e.g., `ate_sbp`, `ate_glucose`) and confidence intervals (if returned by the API)
  - The covariates used (`covariates`)
  - The cleaned participant-level dataframe with CATEs (`cate_df`) for DRLearner runs
  - The CATE column name (`tau_col`)
  - Optional subgroup summaries (`age_effects`, `bmi_effects`)
  - For OLS: the treatment coefficient (`treatment_coef`), its CI, and sample size (`n_obs`)

**What this enables**

Even without opening `econml.API.py`, another user can:

1. Run this notebook,
2. Read the docstrings,
3. Understand exactly how to call each function and what objects they’ll get back.


In [11]:

from econml_utils import build_analysis_df, get_y_t_x

analysis_df = build_analysis_df()
y, t, X, covariates = get_y_t_x(
analysis_df,
outcome_col="sbp_mean",              # or "fasting_glucose_mg_dl"
treatment_col="treatment_supplement"
)


In [12]:
import econml.API as econml_api

sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)
glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")


In [13]:
# -------------------------------------------------------------------
# 8.1  Convenience helpers for summarizing EconML + OLS
# -------------------------------------------------------------------

def summarize_outcome(outcome_col: str, random_state: int = 42):
    """
    Run the DRLearner pipeline and OLS baseline for a single outcome
    and return a compact summary row as a dict.
    """
    if outcome_col == "sbp_mean":
        dr_results = econml_api.run_sbp_supplement_experiment(random_state=random_state)
        ate_key = "ate_sbp"
    elif outcome_col == "fasting_glucose_mg_dl":
        dr_results = econml_api.run_glucose_supplement_experiment(random_state=random_state)
        ate_key = "ate_glucose"
    else:
        # Fall back to the generic pattern: use get_y_t_x + DRLearner setup
        raise ValueError(
            f"Outcome '{outcome_col}' is not wired into summarize_outcome. "
            "Extend this helper if you need more outcomes."
        )

    ols_results = econml_api.run_ols_for_outcome(outcome_col)

    row = {
        "outcome": outcome_col,
        "econml_ate": dr_results[ate_key],
        "ols_treatment_coef": ols_results["treatment_coef"],
        "n_obs_ols": ols_results["n_obs"],
    }
    return row


# Build a summary table for our two outcomes
summary_rows = [
    summarize_outcome("sbp_mean", random_state=42),
    summarize_outcome("fasting_glucose_mg_dl", random_state=42),
]

api_summary_df = pd.DataFrame(summary_rows)

print("API summary (EconML + OLS):")
display(api_summary_df)


API summary (EconML + OLS):


,outcome,econml_ate,ols_treatment_coef,n_obs_ols
0,sbp_mean,-0.077358,1.220973,2638
1,fasting_glucose_mg_dl,-1.886489,-1.611008,2674


# Convenience helper: one summary view for EconML + OLS

So far, we’ve called the API functions one by one:

- `run_sbp_supplement_experiment(...)`
- `run_glucose_supplement_experiment(...)`
- `run_ols_for_outcome(...)`

When you are comparing methods, you usually want the same “headline numbers” each time.
Instead of copy-pasting comparison code, this notebook defines a small **convenience helper**
that collects results in a consistent format.

**What this helper does**

For a given outcome, it:

1. Runs the EconML DRLearner wrapper (to get the ATE).
2. Runs the OLS wrapper (to get the regression treatment coefficient).
3. Returns a compact summary (a single row / dict) with:
   - outcome name
   - EconML ATE (and CI if available)
   - OLS treatment coefficient (and CI if available)
   - number of observations used

**Important note**

This helper is **only for this notebook** (to make the demo cleaner).  
It is **not part of the reusable project API** inside `econml.API.py`.


# Interpreting the API summary

The helper function `summarize_outcome(...)` runs **both** approaches for an outcome:

1. **EconML DRLearner** (causal estimate → ATE)
2. **OLS baseline** (regression coefficient on the treatment variable)

It returns one compact “summary row” per outcome, and we combine those rows into
a small table called `api_summary_df`.

**What each column means**

- **`outcome`**  
  The name of the outcome variable we are modeling (e.g., `sbp_mean`).

- **`econml_ate`**  
  The **Average Treatment Effect (ATE)** from EconML’s DRLearner — interpreted as the
  average change in the outcome when moving from *no supplements* → *any supplements*,
  after adjusting for the covariates.

- **`ols_treatment_coef`**  
  The coefficient on `treatment_supplement` from the OLS regression — interpreted as the
  regression-based estimate of the supplement effect (also controlling for covariates,
  but under a stricter linear model).

- **`n_obs_ols`**  
  The number of observations used in the OLS regression (after dropping missing values).

---

**What we see in this run**

**1) Systolic blood pressure (`sbp_mean`)**

- EconML ATE ≈ **–0.08 mmHg**
- OLS coefficient ≈ **+1.22 mmHg**

Both effects are **small in magnitude**, but the **signs differ**.  
This is a useful teaching example: when effects are tiny, different modeling assumptions
(DRLearner vs. a single linear regression) can produce slightly different point estimates.

**2) Fasting glucose (`fasting_glucose_mg_dl`)**

- EconML ATE ≈ **0**
- OLS coefficient ≈ **0**

Here, both methods agree on a clear **null effect**: with the current sample + covariates,
we do not see a meaningful average impact of “any supplement use” on fasting glucose.

---

**About the FutureWarning messages**

You may see repeated `FutureWarning` messages from pandas during the BMI-bin groupby step.
They relate to a future default change in the `observed=` parameter for `groupby()` and
**do not affect the numeric results** in this notebook.

---

Why this summary is useful (API design)**

`api_summary_df` is the kind of output that is easy to:

- log in tests,
- plot (e.g., EconML vs OLS bar chart),
- or export into a short report,

while the modeling details stay safely inside `econml.API`.



## Summary: Reusing the internal API in other notebooks

This notebook demonstrated the project’s **internal API layer** (not an external data-provider API).  
To reuse the same workflow in a new notebook, follow this pattern.

---

### 1) Build the merged NHANES analysis dataset

```python
from econml_utils import build_analysis_df, get_y_t_x

analysis_df = build_analysis_df()
print(analysis_df.shape)
```

This returns a participant-level dataframe containing:
- outcomes (e.g., `sbp_mean`, `fasting_glucose_mg_dl`)
- the treatment flag (`treatment_supplement`)
- baseline covariates (BMI, lipids, hs-CRP, etc.)

---

### 2) Extract `Y`, `T`, and `X` for a chosen outcome

```python
y, t, X, covariates = get_y_t_x(
    analysis_df,
    outcome_col="sbp_mean",              # or "fasting_glucose_mg_dl"
    treatment_col="treatment_supplement"
)

print(len(y), len(t), X.shape)
print(covariates)
```

Outputs:
- `y`: outcome vector  
- `t`: treatment indicator (0/1)  
- `X`: covariate matrix used for adjustment  
- `covariates`: column names used in `X`

---

### 3) Run DRLearner experiments via `econml.API.py`

Because the file is named `econml.API.py`, load it the same way as this notebook:

```python
import pathlib, sys, importlib.util

api_path = pathlib.Path.cwd() / "econml.API.py"
spec = importlib.util.spec_from_file_location("econml.API", api_path)
econml_api = importlib.util.module_from_spec(spec)
sys.modules["econml.API"] = econml_api
spec.loader.exec_module(econml_api)
```

Then run experiments:

```python
sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)
# If your project exposes this wrapper, you can run it too:
# glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
```

Each results dictionary includes:
- ATE (e.g., `ate_sbp`) and sample size (`n_obs`)
- `cate_df`: cleaned dataframe with individual CATE estimates
- `tau_col`: column name in `cate_df` containing the CATEs
- subgroup summaries (e.g., `bmi_effects`, `age_effects`) when available

Example:

```python
tau_col = sbp_results["tau_col"]
cate_df = sbp_results["cate_df"]
cate_df[[tau_col, "treatment_supplement"]].head()
```

---

### 4) Run the OLS baseline (traditional regression comparison)

```python
ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")
print(ols_sbp)
print(ols_glu)
```

This returns a compact regression estimate (`treatment_coef` + CI + `n_obs`) to compare against the EconML ATE.

---

### Quick reference: API docstrings

In this notebook, we print docstrings for the main entry points so that reusers can see:
1) what each function does, and  
2) exactly what it returns (keys like ATE/CIs, covariates, `cate_df`, `tau_col`, heterogeneity summaries, and `n_obs`).

Final note: `econml.API.ipynb` is the **developer reference**, while `econml.example.ipynb` is the **full narrative tutorial** built on the same API.
